In [2]:
!pip install pyspark


In [2]:
import sqlite3, json, pandas as pd

df = pd.read_csv('week12_transactions.csv')
conn = sqlite3.connect('momo.db')
df.to_sql('transactions', conn, if_exists='replace', index=False)
print(df.shape)
df.head()

(2000, 7)


,txn_id,date,hour,district,txn_type,amount_rwf,flagged
0,T00001,46327,8,Gasabo,send,1000,0
1,T00002,46335,12,Nyarugenge,send,25000,0
2,T00003,46345,8,Muhanga,send,51000,0
3,T00004,46327,17,Huye,send,79000,0
4,T00005,46341,12,Huye,send,109000,0


In [5]:
## Exercises Part 1.1 
# (a) 5 largest 'send' transactions above 100,000 RWF

q =""" 
SELECT txn_id , district , amount_rwf 
FROM transactions 
WHERE txn_type = 'send' AND amount_rwf > 100000
ORDER BY amount_rwf DESC
LIMIT 5
"""
pd.read_sql(q,conn)

,txn_id,district,amount_rwf
0,T01708,Kicukiro,140000
1,T00364,Huye,139000
2,T00895,Muhanga,139000
3,T00899,Nyarugenge,139000
4,T01314,Gasabo,139000


In [7]:
# (b) flagged transactions  (checkpoint: 11)

pd.read_sql ("SELECT COUNT(*) AS flagged_count FROM transactions WHERE flagged =1",conn) 

,flagged_count
0,11


In [13]:
# (a) per txn_type  (checkpoint: send ~62.0M)
q_type = """
SELECT txn_type, 
       COUNT(*) AS n,
       SUM(amount_rwf) AS total_rwf,
       AVG(amount_rwf) AS avg_rwf
FROM transactions
GROUP BY txn_type
ORDER BY total_rwf DESC
"""
print(pd.read_sql(q_type, conn))

  txn_type    n  total_rwf       avg_rwf
0     send  868   62049000  71485.023041
1  paybill  330   29970000  90818.181818
2  receive  489   27024000  55263.803681
3  airtime  313     833000   2661.341853


In [17]:
# (b) per district  (checkpoint: Muhanga ~21.1M)

q_dist = """
SELECT district, 
       SUM(amount_rwf) AS total_rwf
FROM transactions
GROUP BY district
ORDER BY total_rwf DESC
"""
print (pd.read_sql(q_dist, conn))

     district  total_rwf
0     Muhanga   21076000
1      Gasabo   20738000
2        Huye   18736000
3  Nyarugenge   18091000
4    Kicukiro   11338000
5     Musanze   10490500
6      Rubavu   10110500
7   Nyagatare    9296000


In [3]:
# Exercises 1.3
# (a) SQL  (checkpoint: hour 20, 185) 
sql_hours = pd.read_sql (""" SELECT hour,COUNT(*) AS n 
                             FROM transactions 
                             GROUP BY hour 
                             ORDER BY n DESC """,conn)
sql_hours.head()

,hour,n
0,20,185
1,17,183
2,18,176
3,12,170
4,11,169


In [20]:
 ## b) 1.3

pd_hours = df['hour'].value_counts()
print(pd_hours.head())
print('Agree:', pd_hours.index[0] == sql_hours.loc[0,'hour'] and pd_hours.iloc[0] == sql_hours.loc[0,'n'])

hour
20    185
17    183
18    176
12    170
11    169
Name: count, dtype: int64
Agree: True
